# Une boucle PyTorch lisible

**Objectifs.** Vérifier formes et device, obtenir un gradient via autograd, organiser `Dataset`/`DataLoader`, entraîner puis évaluer explicitement. Durée : 60–90 min. Dépendance optionnelle : `pip install -e '.[dl]'`. L’exemple fonctionne sur CPU.

## 1. Tensors, opérations et autograd

Un tensor porte valeurs, forme, dtype et device. Ici `x` est un scalaire. `loss.backward()` applique la règle de chaîne et remplit `weight.grad`; l’optimiseur fera ensuite la mise à jour.

In [ ]:
import torch

torch.set_num_threads(1)
torch.manual_seed(5)
weight = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)
loss = (weight * x - 7) ** 2
loss.backward()
print('loss:', loss.item(), '| dloss/dweight:', weight.grad.item())

## 2. Données et dimensions

La cible suit une relation synthétique `y = 2.5*x - 0.7 + bruit`. Chaque mini-lot a `features: (batch,1)` et `targets: (batch,1)`. Le dernier lot peut être plus petit si le nombre d’exemples n’est pas divisible par batch size.

In [ ]:
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

rng = torch.Generator().manual_seed(8)
features = torch.linspace(-2, 2, 240).unsqueeze(1)
targets = 2.5 * features - 0.7 + 0.15 * torch.randn(features.shape, generator=rng)
dataset = TensorDataset(features, targets)
loader = DataLoader(dataset, batch_size=32, shuffle=True, generator=rng)
print('features:', features.shape, 'targets:', targets.shape, 'batches:', len(loader))

## 3. Modèle, loss et optimizer

`nn.Module` regroupe les paramètres appris. `MSELoss` convient à une cible numérique. SGD applique `parameter -= learning_rate * gradient`. Chaque epoch repasse sur les données d’entraînement.

In [ ]:
model = nn.Linear(in_features=1, out_features=1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.08)

loss_history = []
for epoch in range(80):
    model.train()
    batch_losses = []
    for batch_features, batch_targets in loader:
        prediction = model(batch_features)
        loss = loss_fn(prediction, batch_targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    loss_history.append(sum(batch_losses) / len(batch_losses))
print('weight/bias:', model.weight.item(), model.bias.item())
print('last train MSE:', round(loss_history[-1], 4))

## 4. Évaluation et inference

`eval()` met le module en mode évaluation. `inference_mode()` évite de construire un graphe de gradient inutile. Un dataset test devrait être distinct dans une vraie tâche ; ici les points de démonstration servent uniquement à voir les prédictions apprises.

In [ ]:
model.eval()
with torch.inference_mode():
    probe = torch.tensor([[-1.0], [0.0], [1.0]])
    print('x / prediction:', list(zip(probe.ravel().tolist(), model(probe).ravel().tolist())))

import matplotlib.pyplot as plt
plt.plot(loss_history); plt.xlabel('Epoch'); plt.ylabel('Train MSE')
plt.title('Boucle PyTorch'); plt.grid(alpha=0.2); plt.show()

## 5. Sauvegarde, CPU/GPU et reproductibilité

En pratique, enregistrer `model.state_dict()` et documenter les dimensions d’entrée, transformations, versions, seed et split. Pour GPU, sélectionner `device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')`, puis transférer modèle et lots. Les calculs GPU et certains noyaux peuvent varier entre machines.

In [ ]:
import io

buffer = io.BytesIO()
torch.save(model.state_dict(), buffer)
buffer.seek(0)
restored = nn.Linear(1, 1)
restored.load_state_dict(torch.load(buffer, weights_only=True))
print('restored prediction:', restored(torch.tensor([[0.0]])).item())

## À retenir et exercice

La séquence essentielle est mode train → forward → loss → zero_grad → backward → optimizer step. En validation : mode eval + inference_mode. Pour aller plus loin, séparez train/validation, ajoutez un DataLoader de validation, comparez Adam à SGD et tracez les deux courbes. Ce mini-problème ne démontre pas une performance de généralisation.